In [1]:
# 09_leave_domain_out — Cell 1: random split vs domain-held-out split (TF-IDF + LR), c2020 & c2025
import os, pandas as pd, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score

P = "/mnt/g/banglafake-detection/data/processed/v2"
R = "/mnt/g/banglafake-detection/reports/v2"; os.makedirs(R, exist_ok=True)
SEED = 0

CORP = {n: pd.read_csv(f"{P}/{n}.csv") for n in ("c2020", "c2025")}
for df in CORP.values():
    df["text"] = df["text"].fillna("").str[:1500]
    df["domain"] = df["domain"].fillna("unk")

def ev(y, p, s):
    return dict(bacc=round(balanced_accuracy_score(y, p), 3),
                f1=round(f1_score(y, p, average="macro"), 3),
                auc=round(roc_auc_score(y, s), 3))

def fit_eval(tr, te, tag):
    v = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=200000,
                        sublinear_tf=True, min_df=3)
    clf = LogisticRegression(max_iter=300, class_weight="balanced", C=2.0)
    clf.fit(v.fit_transform(tr["text"]), tr["label"])
    s = clf.decision_function(v.transform(te["text"]))
    p = (s > 0).astype(int)
    return dict(split=tag, n_train=len(tr), n_test=len(te),
                n_domains_train=tr["domain"].nunique(), n_domains_test=te["domain"].nunique(),
                **ev(te["label"], p, s))

rows = []
for name, df in CORP.items():
    # (a) random 70/30 stratified — domains free to repeat between train/test
    tr, te = train_test_split(df, test_size=0.3, stratify=df["label"], random_state=SEED)
    rows.append(dict(corpus=name, **fit_eval(tr, te, "random_split")))

    # (b) domain-held-out — GroupShuffleSplit guarantees zero domain overlap
    gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED)
    tr_idx, te_idx = next(gss.split(df, groups=df["domain"]))
    tr, te = df.iloc[tr_idx], df.iloc[te_idx]
    overlap = len(set(tr["domain"]) & set(te["domain"]))
    rows.append(dict(corpus=name, **fit_eval(tr, te, "domain_heldout"), domain_overlap=overlap))

res = pd.DataFrame(rows)
res.to_csv(f"{R}/leave_domain_out.csv", index=False)
print(res.to_string(index=False))

corpus          split  n_train  n_test  n_domains_train  n_domains_test  bacc    f1   auc  domain_overlap
 c2020   random_split    34874   14947               81              51 0.949 0.903 0.993             NaN
 c2020 domain_heldout    34303   15518               65              28 0.768 0.812 0.969             0.0
 c2025   random_split     2785    1194               21              18 0.960 0.960 0.992             NaN
 c2025 domain_heldout     3280     699               18               8 0.589 0.535 0.688             0.0


In [2]:
# 09_leave_domain_out — Cell 2: random split vs domain-held-out (BanglaBERT), c2020 & c2025
import os, pandas as pd, torch, torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score

P = "/mnt/g/banglafake-detection/data/processed/v2"
R = "/mnt/g/banglafake-detection/reports/v2"; os.makedirs(R, exist_ok=True)
MODEL = "csebuetnlp/banglabert"
MAXLEN = 256
SEED = 0

try:
    from normalizer import normalize as bn_normalize
except ImportError:
    bn_normalize = lambda s: s

CORP = {n: pd.read_csv(f"{P}/{n}.csv") for n in ("c2020", "c2025")}
for df in CORP.values():
    df["text"] = df["text"].fillna("").str[:2000].map(bn_normalize)
    df["domain"] = df["domain"].fillna("unk")

tok = AutoTokenizer.from_pretrained(MODEL, use_fast=False)

class TxtDS(torch.utils.data.Dataset):
    def __init__(self, df):
        self.enc = tok(df["text"].tolist(), truncation=True, max_length=MAXLEN, padding=False)
        self.labels = df["label"].astype(int).tolist()
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item

def ev(y, p, s):
    return dict(bacc=round(balanced_accuracy_score(y, p), 3),
                f1=round(f1_score(y, p, average="macro"), 3),
                auc=round(roc_auc_score(y, s), 3))

class WeightedTrainer(Trainer):
    def __init__(self, *a, class_weights=None, **kw):
        super().__init__(*a, **kw)
        self.cw = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = nn.functional.cross_entropy(out.logits, labels, weight=self.cw.to(out.logits.device))
        return (loss, out) if return_outputs else loss

def fit_eval(tr_df, te_df, tag, corpus):
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
    vc = tr_df["label"].value_counts()
    cw = torch.tensor([len(tr_df) / (2 * vc.get(0, 1)), len(tr_df) / (2 * vc.get(1, 1))], dtype=torch.float)
    args = TrainingArguments(
        output_dir="/tmp/bb_run", per_device_train_batch_size=16, per_device_eval_batch_size=64,
        num_train_epochs=3, learning_rate=2e-5, weight_decay=0.01, logging_steps=200,
        save_strategy="no", report_to=[], seed=SEED, fp16=torch.cuda.is_available(),
    )
    trainer = WeightedTrainer(model=model, args=args, train_dataset=TxtDS(tr_df),
                               data_collator=lambda feats: tok.pad(feats, return_tensors="pt"),
                               class_weights=cw)
    trainer.train()
    logits = trainer.predict(TxtDS(te_df)).predictions
    s = logits[:, 1] - logits[:, 0]
    p = (s > 0).astype(int)
    return dict(corpus=corpus, split=tag, n_train=len(tr_df), n_test=len(te_df),
                n_domains_train=tr_df["domain"].nunique(), n_domains_test=te_df["domain"].nunique(),
                **ev(te_df["label"].values, p, s))

rows = []
for name, df in CORP.items():
    tr, te = train_test_split(df, test_size=0.3, stratify=df["label"], random_state=SEED)
    rows.append(fit_eval(tr, te, "random_split", name))

    gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED)
    tr_idx, te_idx = next(gss.split(df, groups=df["domain"]))
    tr, te = df.iloc[tr_idx], df.iloc[te_idx]
    r = fit_eval(tr, te, "domain_heldout", name)
    r["domain_overlap"] = len(set(tr["domain"]) & set(te["domain"]))
    rows.append(r)

res = pd.DataFrame(rows)
res.to_csv(f"{R}/leave_domain_out_bert.csv", index=False)
print(res.to_string(index=False))

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
200,0.461100
400,0.341200
600,0.299000
800,0.328800
1000,0.329200
1200,0.308400
1400,0.189900
1600,0.334000
1800,0.293300
2000,0.236200


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
200,0.467900
400,0.356600
600,0.308700
800,0.224300
1000,0.267000
1200,0.213700
1400,0.244200
1600,0.211200
1800,0.205400
2000,0.198300


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
200,0.182900
400,0.045500


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
200,0.196900
400,0.044800
600,0.028000


corpus          split  n_train  n_test  n_domains_train  n_domains_test  bacc    f1   auc  domain_overlap
 c2020   random_split    34874   14947               81              51 0.938 0.953 0.992             NaN
 c2020 domain_heldout    34303   15518               65              28 0.754 0.828 0.955             0.0
 c2025   random_split     2785    1194               21              18 0.981 0.981 0.995             NaN
 c2025 domain_heldout     3280     699               18               8 0.659 0.631 0.801             0.0
